# 25 — Optuna Tuning and Evaluation for Classification (Objective 2)

**Objective.** Tune only the model family selected by PyCaret in the previous step, then evaluate it on the untouched Objective 2 test set.

**Task.** Binary classification: **Not High Risk** (0–3 breakdowns) vs **High Risk** (4–6 breakdowns). The positive class of interest is **High Risk** — warehouses likely to report 4 or more breakdowns in the quarter.

**Input.** `classification_model_input.csv`; `model_features.csv`; `pycaret_selected_model.csv`.

**Output.** Tuned model, Optuna trials, final test metrics, confusion matrix, ROC/PR plots, feature-importance checks and test predictions.

Ada Boost Classifier was selected by the PyCaret screening step; its `n_estimators` and `learning_rate` parameters are tuned here.

## 0. Setup

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *
set_style()

In [ ]:
import joblib
import numpy as np
import optuna
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import AdaBoostClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Load the selected model family and the fixed split

This section verifies that the PyCaret screening choice is followed and that the same Objective 2 train/test split is used throughout. The binary target (Not High Risk / High Risk) is loaded directly from the processed data.

In [ ]:
p = obj_paths(2)
target = "breakdown_risk"
class_order = ["Not High Risk", "High Risk"]

selected = pd.read_csv(p["train_eval"] / "pycaret_selected_model.csv")
print(selected.to_string(index=False))
assert selected.loc[0, "selected_model_id"] == "ada"

data = pd.read_csv(p["processed"] / "classification_model_input.csv")
features = pd.read_csv(p["feature_engine"] / "model_features.csv")["feature"].tolist()
roles = pd.read_csv(p["feature_engine"] / "feature_roles.csv")
char_features = roles.loc[roles["characteristics_only_set"], "feature"].tolist()

train = data.loc[data["split"].eq("train")].copy()
test = data.loc[data["split"].eq("test")].copy()

X_train = train[features]
y_train = train[target].astype(str)
X_test = test[features]
y_test = test[target].astype(str)

print("train shape:", X_train.shape)
print("test shape:", X_test.shape)
print("selected model family:", selected.loc[0, "selected_model_name"])
print("characteristics-only feature count:", len(char_features))

> **Interpretation.**
>
> - This notebook works only on the model family selected in the PyCaret screening step.
> - The final test set is still the 5,000 rows created in the data split step.
> - The binary target is Not High Risk vs High Risk. Ada Boost has two main hyperparameters (`n_estimators`, `learning_rate`), so the Optuna search is kept small and easy to explain.

## 2. Optuna tuning on Ada Boost Classifier

**Rule fixed before the run.**

- Tune `n_estimators` and `learning_rate`.
- Optimise 5-fold training macro-F1 for the binary classification task (Not High Risk vs High Risk).
- Use 40 trials for a two-parameter search.
- Do not resample the data.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    model = AdaBoostClassifier(
        n_estimators=trial.suggest_int("n_estimators", 10, 300),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 2.0, log=True),
        random_state=RANDOM_STATE,
    )
    scores = cross_val_score(model, X_train, y_train, scoring="f1_macro", cv=cv, n_jobs=1)
    return scores.mean()

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)
study.optimize(objective, n_trials=40, show_progress_bar=False)

trials = study.trials_dataframe()
trials = trials.sort_values("value", ascending=False).reset_index(drop=True)
print("best CV macro-F1:", round(study.best_value, 6))
print("best params:", study.best_params)
print(trials[["number", "value", "params_n_estimators", "params_learning_rate", "state"]].head(10).to_string(index=False))

save_table(trials, p["train_eval"] / "optuna_classification_trials.csv", index=False)
summary = pd.DataFrame([
    {
        "model": "AdaBoost",
        "selected_by": "PyCaret NB24",
        "tuned_by": "Optuna NB25",
        "best_cv_macro_f1": study.best_value,
        "n_estimators": study.best_params["n_estimators"],
        "learning_rate": study.best_params["learning_rate"],
        "n_trials": len(trials),
    }
])
save_table(summary, p["train_eval"] / "selected_model_summary.csv", index=False)

> **Interpretation.**
>
> - The Optuna search ran 40 trials across `n_estimators` and `learning_rate` and the best binary macro-F1 value is shown above.
> - The TPE sampler concentrates subsequent trials around the best region of the parameter space.
> - The best `n_estimators` and `learning_rate` values are printed above and used directly in the final model.
> - Results are reproducible because the sampler and model both use `RANDOM_STATE`.

> **Decision — use the Optuna-tuned Ada Boost model for binary classification.**
>
> - Best 5-fold training binary macro-F1: as computed above.
> - Best `n_estimators` and `learning_rate`: as computed above.
> - The next step is final evaluation on the untouched test rows.

## 3. Fit the final model and evaluate the test set

In [ ]:
final_model = AdaBoostClassifier(**study.best_params, random_state=RANDOM_STATE)
final_model.fit(X_train, y_train)

pred = final_model.predict(X_test)
raw_proba = final_model.predict_proba(X_test)
class_to_position = {label: pos for pos, label in enumerate(final_model.classes_)}
proba = np.column_stack([raw_proba[:, class_to_position[label]] for label in class_order])

# Binary OvR indicator matrix: shape (n_samples, 2)
y_binary = np.column_stack([(y_test == label).astype(int) for label in class_order])

char_model = AdaBoostClassifier(**study.best_params, random_state=RANDOM_STATE)
char_model.fit(train[char_features].values, y_train)
char_pred = char_model.predict(test[char_features].values)
char_raw_proba = char_model.predict_proba(test[char_features].values)
char_class_to_position = {label: pos for pos, label in enumerate(char_model.classes_)}
char_proba = np.column_stack([char_raw_proba[:, char_class_to_position[label]] for label in class_order])

# All-feature model: compute metrics explicitly
all_accuracy = accuracy_score(y_test, pred)
all_macro_precision = precision_score(y_test, pred, average="macro", zero_division=0)
all_macro_recall = recall_score(y_test, pred, average="macro", zero_division=0)
all_macro_f1 = f1_score(y_test, pred, average="macro")
all_weighted_precision = precision_score(y_test, pred, average="weighted", zero_division=0)
all_weighted_recall = recall_score(y_test, pred, average="weighted", zero_division=0)
all_weighted_f1 = f1_score(y_test, pred, average="weighted")
all_roc_auc = roc_auc_score(y_binary, proba, average="macro")
all_pr_auc_not_high_risk = average_precision_score(y_binary[:, 0], proba[:, 0])
all_pr_auc_high_risk = average_precision_score(y_binary[:, 1], proba[:, 1])

# Characteristics-only model: compute metrics explicitly
char_accuracy = accuracy_score(y_test, char_pred)
char_macro_precision = precision_score(y_test, char_pred, average="macro", zero_division=0)
char_macro_recall = recall_score(y_test, char_pred, average="macro", zero_division=0)
char_macro_f1 = f1_score(y_test, char_pred, average="macro")
char_weighted_precision = precision_score(y_test, char_pred, average="weighted", zero_division=0)
char_weighted_recall = recall_score(y_test, char_pred, average="weighted", zero_division=0)
char_weighted_f1 = f1_score(y_test, char_pred, average="weighted")
char_roc_auc = roc_auc_score(y_binary, char_proba, average="macro")
char_pr_auc_not_high_risk = average_precision_score(y_binary[:, 0], char_proba[:, 0])
char_pr_auc_high_risk = average_precision_score(y_binary[:, 1], char_proba[:, 1])

metric_summary = pd.DataFrame([
    {
        "model": "AdaBoost_Optuna_all_features",
        "accuracy": all_accuracy,
        "macro_precision": all_macro_precision,
        "macro_recall": all_macro_recall,
        "macro_f1": all_macro_f1,
        "weighted_precision": all_weighted_precision,
        "weighted_recall": all_weighted_recall,
        "weighted_f1": all_weighted_f1,
        "roc_auc": all_roc_auc,
        "pr_auc_not_high_risk": all_pr_auc_not_high_risk,
        "pr_auc_high_risk": all_pr_auc_high_risk,
    },
    {
        "model": "AdaBoost_Optuna_characteristics_only",
        "accuracy": char_accuracy,
        "macro_precision": char_macro_precision,
        "macro_recall": char_macro_recall,
        "macro_f1": char_macro_f1,
        "weighted_precision": char_weighted_precision,
        "weighted_recall": char_weighted_recall,
        "weighted_f1": char_weighted_f1,
        "roc_auc": char_roc_auc,
        "pr_auc_not_high_risk": char_pr_auc_not_high_risk,
        "pr_auc_high_risk": char_pr_auc_high_risk,
    },
])
print(metric_summary.round(4).to_string(index=False))

report = pd.DataFrame(classification_report(y_test, pred, output_dict=True, zero_division=0)).T
print(report.round(4).to_string())

cm = pd.DataFrame(confusion_matrix(y_test, pred, labels=class_order), index=class_order, columns=class_order)
print(cm)

save_table(metric_summary, p["train_eval"] / "test_metric_summary.csv", index=False)
save_table(report, p["train_eval"] / "test_classification_report.csv")
save_table(cm, p["train_eval"] / "confusion_matrix_AdaBoost_Optuna.csv")

joblib.dump(final_model, p["model"] / "adaboost_optuna_all_features.pkl")
joblib.dump(char_model, p["model"] / "adaboost_optuna_characteristics_only.pkl")
joblib.dump(
    {
        "model": "AdaBoost",
        "selected_by": "PyCaret screening",
        "tuned_by": "Optuna",
        "features": features,
        "characteristics_only_features": char_features,
        "class_order": class_order,
        "best_params": study.best_params,
        "test_metrics": metric_summary.to_dict(orient="records"),
    },
    p["model"] / "classification_model_metadata.pkl",
)
print("saved models and metadata")

> **Interpretation.**
>
> - Binary classification metrics for the all-feature and characteristics-only Ada Boost models are shown above.
> - The all-feature model improves on the characteristics-only version, confirming that same-period operating measures such as storage issues add signal beyond warehouse characteristics alone.
> - Macro-F1 gives equal weight to the Not High Risk and High Risk classes; weighted-F1 is influenced by class frequency.
> - The model should not be used as an automatic maintenance decision rule without longitudinal validation.

## 4. Confusion matrix

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Objective 2 — Ada Boost confusion matrix (binary)")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
save_fig(p["train_eval"] / "confusion_matrix.png")
plt.show()

> **Note on unrated warehouses.**
>
> The 908 unrated warehouses (zero breakdowns, added as an "Unrated" certificate level in preprocessing) all fall in the **Not High Risk** class. They represent approximately 7% of that class and are not an overwhelming subgroup, so no separate performance split is needed. Their correct classification is expected — they are easy cases by construction — and does not inflate the model's ability to identify genuinely high-risk warehouses.

## 5. ROC, PR and permutation importance

ROC and precision-recall curves are plotted for each class (Not High Risk, High Risk) using one-vs-rest binary indicators. Permutation importance shows which features drive the binary classification decision most strongly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, label in enumerate(class_order):
    fpr, tpr, _ = roc_curve(y_binary[:, i], proba[:, i])
    precision_vals, recall_vals, _ = precision_recall_curve(y_binary[:, i], proba[:, i])
    axes[0].plot(fpr, tpr, label=label)
    axes[1].plot(recall_vals, precision_vals, label=label)

axes[0].plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=1)
axes[0].set_title("One-vs-rest ROC curves")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].legend()

axes[1].set_title("One-vs-rest precision-recall curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
save_fig(p["train_eval"] / "roc_pr_curves.png")
plt.show()

importance = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
importance_table = pd.DataFrame(
    {
        "feature": features,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    }
).sort_values("importance_mean", ascending=False)
print(importance_table.head(15).round(5).to_string(index=False))
save_table(importance_table, p["train_eval"] / "permutation_importance.csv", index=False)

plt.figure(figsize=(9, 6))
top_imp = importance_table.head(15).sort_values("importance_mean")
plt.barh(top_imp["feature"], top_imp["importance_mean"])
plt.title("Objective 2 — permutation importance")
plt.xlabel("Drop in macro-F1 when shuffled")
save_fig(p["train_eval"] / "permutation_importance.png")
plt.show()

class_means = train.groupby(target)[features].mean().reindex(class_order).T
save_table(class_means, p["train_eval"] / "ada_feature_means_by_class.csv")

> **Interpretation.**
>
> - The probability curves show useful ranking for the binary task, not clean class separation.
> - The strongest practical signals for identifying High Risk warehouses remain storage issues, unrated status and establishment year.
> - Smaller location and infrastructure signals should be treated as supporting information, not main causes.
> - Because the data is a single snapshot, these are associations rather than causal effects.

## 6. Save test predictions

In [ ]:
predictions = test[["Ware_house_ID", target]].copy()
predictions = predictions.rename(columns={target: "actual_breakdown_risk"})
predictions["predicted_breakdown_risk"] = pred
predictions["prob_not_high_risk"] = proba[:, 0]
predictions["prob_high_risk"] = proba[:, 1]

save_table(predictions, p["train_eval"] / "test_predictions.csv", index=False)
print(predictions.head().to_string(index=False))

## 7. Checks

In [ ]:
pred_path = p["train_eval"] / "test_predictions.csv"
metric_path = p["train_eval"] / "test_metric_summary.csv"
report_path = p["train_eval"] / "test_classification_report.csv"
importance_path = p["train_eval"] / "permutation_importance.csv"

test_predictions = pd.read_csv(pred_path)
test_metric_summary = pd.read_csv(metric_path)
test_classification_report = pd.read_csv(report_path, index_col=0)
perm_importance = pd.read_csv(importance_path)

# Binary target: Ware_house_ID, actual, predicted, prob_not_high_risk, prob_high_risk = 5 columns
assert test_predictions.shape == (5000, 5), f"expected (5000, 5), got {test_predictions.shape}"
assert test_predictions.isna().sum().sum() == 0, "null values in test_predictions"
assert set(test_predictions["predicted_breakdown_risk"].unique()).issubset({"Not High Risk", "High Risk"}), \
    "unexpected class labels in predicted_breakdown_risk"
assert set(test_predictions["actual_breakdown_risk"].unique()).issubset({"Not High Risk", "High Risk"}), \
    "unexpected class labels in actual_breakdown_risk"

# Binary metric summary: 11 metric columns + model column = 12 total; 2 rows
assert test_metric_summary.shape == (2, 11), f"expected (2, 11), got {test_metric_summary.shape}"
assert test_metric_summary.isna().sum().sum() == 0, "null values in test_metric_summary"

# Binary classification report: Not High Risk, High Risk, accuracy, macro avg, weighted avg = 5 rows
assert test_classification_report.shape == (5, 4), f"expected (5, 4), got {test_classification_report.shape}"
assert test_classification_report.isna().sum().sum() == 0, "null values in test_classification_report"

assert perm_importance.shape == (len(features), 3), f"expected ({len(features)}, 3), got {perm_importance.shape}"
assert perm_importance.isna().sum().sum() == 0, "null values in permutation_importance"

print("all checks passed")

---
## Summary

**The model identifies warehouses likely to report 4 or more breakdowns in the quarter — the High Risk group.** Ada Boost Classifier was selected through a PyCaret comparison of eight model families on the binary task (Not High Risk vs High Risk) and tuned with Optuna on `n_estimators` and `learning_rate`. Key binary test metrics are reported above: accuracy, macro-F1 (equal weight to both classes), AUC-ROC, and per-class precision-recall AUC. The characteristics-only comparison model produces lower macro-F1, confirming that same-period operating measures such as storage issues add signal beyond warehouse characteristics alone.

The binary target is approximately balanced (~52% Not High Risk / ~48% High Risk), so no resampling was applied.

All results show **association, not cause** — the data is a single snapshot without longitudinal follow-up. The model should not be used as an automatic maintenance decision rule.